# 🛡️ Level 9: Regularization (Ridge & Lasso)

**[📖 Want a detailed explanation? Read the Manual (Streamlit App)](https://bookseal-seoul-apt-price-prediction.streamlit.app/Level_9_Regularization)**

When we add too many features (polynomials), the model **overfits** (memorizes the data).
**Regularization** punishes the model for being too complex.

- **Ridge (L2)**: Shrinks all coefficients (good for correlated features).
- **Lasso (L1)**: Sets useless coefficients to ZERO (feature selection).

### 💡 Mental Model: The Complexity Budget

Imagine every feature you use costs money (Penalty).
- **Linear Regression**: Unlimited budget. Buy everything! (Overfits)
- **Regularization**: You have a limited budget ($Alpha*). Use only what matters.
    - **Ridge**: Buy cheaper versions of everything.
    - **Lasso**: Buy only the essential items and drop the rest.

### 1. Load Data & Create Polynomial Features
We will intentionally create **too many features** to force overfitting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

url = "https://github.com/bookseal/seoul-apt-price-prediction/raw/main/data/sample.parquet"
df = pd.read_parquet(url)

# Create X and y
X = df[['area_m2', 'year', 'floor']].values
y = df['price_10k_krw'].values

# Add Polynomial Features (Degree 3 -> Many features!)
poly = PolynomialFeatures(degree=3, include_bias=False)
X_poly = poly.fit_transform(X)

# Scale (Important for Regularization!)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_poly)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print(f"Original Features: {X.shape[1]}")
print(f"Polynomial Features: {X_poly.shape[1]}")

### 2. Linear Regression (No Penalty)
Watch it overfit (or be unstable).

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

train_rmse = np.sqrt(mean_squared_error(y_train, lr.predict(X_train)))
test_rmse = np.sqrt(mean_squared_error(y_test, lr.predict(X_test)))

print(f"Linear Regression - Train RMSE: {train_rmse:,.0f} | Test RMSE: {test_rmse:,.0f}")
print(f"Sum of Coefficients: {np.sum(np.abs(lr.coef_)):,.0f}")

### 3. Ridge Regression (L2 Penalty)
Shrinks coefficients.

In [ ]:
ridge = Ridge(alpha=10.0)
ridge.fit(X_train, y_train)

train_rmse = np.sqrt(mean_squared_error(y_train, ridge.predict(X_train)))
test_rmse = np.sqrt(mean_squared_error(y_test, ridge.predict(X_test)))

print(f"Ridge (alpha=10) - Train RMSE: {train_rmse:,.0f} | Test RMSE: {test_rmse:,.0f}")
print(f"Sum of Coefficients: {np.sum(np.abs(ridge.coef_)):,.0f}")

### 4. Lasso Regression (L1 Penalty)
Selects features (sets irrelevant ones to 0).

In [ ]:
lasso = Lasso(alpha=10.0, max_iter=10000)
lasso.fit(X_train, y_train)

train_rmse = np.sqrt(mean_squared_error(y_train, lasso.predict(X_train)))
test_rmse = np.sqrt(mean_squared_error(y_test, lasso.predict(X_test)))

print(f"Lasso (alpha=10) - Train RMSE: {train_rmse:,.0f} | Test RMSE: {test_rmse:,.0f}")
print(f"Non-zero features: {np.sum(lasso.coef_ != 0)} / {X_train.shape[1]}")